# 03 — Pre-DeFi Econometric Core Panel (Report)

Reading-side companion to `run_core_panel.py`. The script merges the
master calendar, Bybit perp data (klines / OI / funding) and the spot
BTC/ETH benchmarks into the canonical hourly panel and dumps two
artefacts:

- `data/econ/econ_core_predefi_1h.parquet` — the 22-column, 41 328-row panel
- `data/analysis/reports/econ_core_predefi_qa.json` — missing-data audit

This notebook reads those files and verifies the panel against the
downstream contract without re-running any computation. This report
supersedes the original exploratory notebook (`03_core_panel.ipynb`),
which is not part of the replication package.

**Contract with NB04.** `econ_core_predefi_1h.parquet` schema is locked:
22 columns, in this exact order, with `liq_usd_total` and `log_liq` as
all-NaN placeholders that NB04 will fill. Any drift breaks NB04 and,
transitively, NB07 / NB08 / NB09.

## 0. Setup & auto-execution

In [ ]:
# ── Setup ──
import sys, json, subprocess, time
from pathlib import Path
sys.path.insert(0, "..")
from config import CFG, ECON_DIR, REPORTS_DIR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Toggles
FORCE_RERUN = False    # if True, regenerate all outputs via the script

SCRIPT  = Path.cwd().parent / "scripts" / "run_core_panel.py"
OUTPUTS = {
    "panel": ECON_DIR    / "econ_core_predefi_1h.parquet",
    "qa":    REPORTS_DIR / "econ_core_predefi_qa.json",
}

print("Setup OK")
print(f"  FORCE_RERUN = {FORCE_RERUN}")

In [ ]:
# ── Auto-execute if outputs are missing or FORCE_RERUN is set ──
missing = [t for t, p in OUTPUTS.items() if not p.exists()]

if FORCE_RERUN:
    print("FORCE_RERUN=True → re-running run_core_panel.py...", flush=True)
    run = True
elif missing:
    print(f"Outputs missing ({missing}) → running run_core_panel.py...", flush=True)
    run = True
else:
    print("All outputs present, skipping recompute (FORCE_RERUN=False)")
    for t, p in OUTPUTS.items():
        print(f"  [{t}] {p.name}")
    run = False

if run:
    assert SCRIPT.exists(), SCRIPT
    cmd = [sys.executable, str(SCRIPT)]
    print("Running:", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=str(Path.cwd().parent), check=True)
    print(f"\nScript done — {time.time()-t0:.1f}s")

In [ ]:
# ── Load artefacts ──
df = pd.read_parquet(OUTPUTS["panel"], engine="pyarrow")
df["date"] = pd.to_datetime(df["date"], utc=True)
with open(OUTPUTS["qa"]) as f:
    qa = json.load(f)

print(f"panel: {len(df):,} rows × {df.shape[1]} cols")
print(f"qa:    status = {qa['status']}")

## 1. Description du panel produit

Bornes temporelles, taille du fichier, et inventaire des colonnes
regroupées par fonction. Doit afficher 41 328 lignes × 22 colonnes
sur la fenêtre `[2021-03-15, 2025-12-01)`.

In [ ]:
print("═══ Description du panel ═══\n")
size_mb = OUTPUTS["panel"].stat().st_size / 1e6
print(f"  Rows:       {len(df):,}")
print(f"  Cols:       {df.shape[1]}")
print(f"  Date range: [{df['date'].min()}, {df['date'].max()}]")
print(f"  Frequency:  {CFG.FREQ}")
print(f"  File size:  {size_mb:.2f} MB")

col_groups = {
    "Identifier":      ["date"],
    "Prices":          ["close_perp", "close_btc_spot", "close_eth_spot"],
    "Volume":          ["volume_perp"],
    "Returns":         ["ret_eth_perp", "ret_btc_spot", "ret_eth_spot"],
    "OI features":     ["oi", "d_oi", "oi_zscore", "oi_high", "oi_vol_ratio"],
    "Volatility":      ["vol_eth_7d", "vol_btc_7d", "ret_eth_std", "ret_btc_std"],
    "Funding & basis": ["funding_rate", "funding_high", "basis_bps"],
    "DeFi (placeholders, filled by NB04)": ["liq_usd_total", "log_liq"],
}
print("\n── Columns ──")
for group, cols in col_groups.items():
    present = [c for c in cols if c in df.columns]
    print(f"  {group:42s}: {present}")

## 2. Missing rates par colonne

Les colonnes-clé (`close_perp`, `oi`, `funding_rate`, `close_btc_spot`,
`close_eth_spot`) doivent être à 0 % de manquants. Les colonnes dérivées
à fenêtre roulante (vol, oi_zscore) sont attendues NaN sur la période
de warmup. Les placeholders DeFi (`liq_usd_total`, `log_liq`) sont à
100 % NaN — c'est le comportement contractuel.

In [ ]:
miss = (df.isna().sum() / len(df) * 100).round(3)
miss = miss.sort_values(ascending=False).rename("missing_pct")
print("═══ Missing rates (%) ═══\n")
print(miss.to_string())

## 3. Statistiques sommaires des principales variables

`describe()` sur les six variables qui pilotent toute l'analyse aval.

In [ ]:
main_vars = ["close_perp", "ret_eth_perp", "oi",
             "funding_rate", "basis_bps", "vol_eth_7d"]
stats = df[main_vars].describe().T.round(4)
print("═══ Summary stats ═══\n")
print(stats.to_string())

## 4. Contrôle des placeholders DeFi

`liq_usd_total` et `log_liq` doivent être **intégralement NaN** à ce
stade (NB04 les remplira). Toute valeur non-NaN ici est une régression.

In [ ]:
print("═══ Placeholder check ═══\n")
for col in ("liq_usd_total", "log_liq"):
    n_nan = df[col].isna().sum()
    ok = n_nan == len(df)
    flag = "OK" if ok else "FAIL"
    print(f"  [{flag}] {col}: {n_nan:,}/{len(df):,} NaN")
    assert ok, f"{col} has non-NaN values at the NB03 stage"

## 5. Sanity check — alignement temporel des sources

`basis_bps = 1e4 · (close_perp − close_eth_spot) / close_eth_spot` est
la seule quantité qui combine les deux sources distinctes (Bybit perp
+ CCData spot ETH). Un désalignement temporel d'une heure ou un fuseau
incorrect dans le merge fait diverger visuellement la série. Si elle
oscille autour de 0 avec quelques spikes ponctuels, le merge est sain.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.2))
ax.plot(df["date"], df["basis_bps"], color="black", linewidth=0.5)
ax.axhline(0, color="black", linestyle=":", linewidth=0.6)
ax.set_xlabel("")
ax.set_ylabel("basis (bps)")
ax.margins(x=0)
plt.tight_layout()
plt.show()

## Next step

- **Notebook 04 (`04_defi_merge.ipynb`)** consumes
  `econ_core_predefi_1h.parquet`, merges the DeFi liquidation feed,
  computes `log_liq`, `log_liq_lag1`, `liq_stress`, `shock_x_oi`, and
  writes `econ_core_full_1h.parquet` — the file actually consumed by
  NB07 / NB08 / NB09 and the factorised scripts.